# Category 6 - ASR Transcription Comparison

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares ASR stacks for the speech-input layer in Ally Vision v2. The project’s most important criteria are real-time streaming fit, multilingual Indian-language handling, interruption compatibility, and platform integration — not just a single English WER leaderboard line.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
# Hardcoded from public-source URLs and project-grounded measurements only.
# No runtime web calls are performed in this notebook.
data = {
  "source_urls": {
    "Whisper large-v3 HF card": "https://huggingface.co/openai/whisper-large-v3",
    "Google STT v2 docs": "https://cloud.google.com/speech-to-text/v2/docs",
    "Azure Speech language support": "https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt",
    "Deepgram model docs": "https://developers.deepgram.com/docs/model",
    "Project realtime client": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py"
  },
  "comparison_rows": [
    {
      "Metric": "Streaming / realtime support",
      "gummy-realtime-v1 / qwen3-asr-flash-realtime": "Yes, unified inside the realtime session [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "Whisper large-v3": "No native realtime out of the box [source: https://huggingface.co/openai/whisper-large-v3]",
      "Google Speech-to-Text v2": "Yes, streaming audio supported [source: https://cloud.google.com/speech-to-text/v2/docs]",
      "Azure Speech": "Yes, real-time transcription supported [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Deepgram Nova-2": "Yes, streaming models documented [source: https://developers.deepgram.com/docs/model]",
      "AssemblyAI Universal-2": "N/A (not publicly available) [source: https://www.assemblyai.com/docs]",
      "Amazon Transcribe": "Yes [source: https://docs.aws.amazon.com/transcribe/]",
      "Sarvam AI Saarika": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "AI4Bharat IndicConformer": "N/A (not publicly available) [source: https://huggingface.co/ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large]",
      "Speechmatics Real-time": "Yes [source: https://docs.speechmatics.com/]",
      "Kaldi / ESPnet": "Possible with custom serving [source: https://github.com/espnet/espnet]"
    },
    {
      "Metric": "English WER baseline",
      "gummy-realtime-v1 / qwen3-asr-flash-realtime": "N/A exact public English WER not captured [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "Whisper large-v3": "Mean WER 7.44 on HF open ASR leaderboard [source: https://huggingface.co/openai/whisper-large-v3]",
      "Google Speech-to-Text v2": "N/A (not publicly available) [source: https://cloud.google.com/speech-to-text/v2/docs]",
      "Azure Speech": "N/A (not publicly available) [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Deepgram Nova-2": "Vendor page claims major WER improvement over competitors [source: https://developers.deepgram.com/docs/model]",
      "AssemblyAI Universal-2": "N/A (not publicly available) [source: https://www.assemblyai.com/docs]",
      "Amazon Transcribe": "N/A (not publicly available) [source: https://docs.aws.amazon.com/transcribe/]",
      "Sarvam AI Saarika": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "AI4Bharat IndicConformer": "N/A (not publicly available) [source: https://huggingface.co/ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large]",
      "Speechmatics Real-time": "N/A (not publicly available) [source: https://docs.speechmatics.com/]",
      "Kaldi / ESPnet": "N/A (not publicly available) [source: https://github.com/espnet/espnet]"
    },
    {
      "Metric": "Hindi WER baseline",
      "gummy-realtime-v1 / qwen3-asr-flash-realtime": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "Whisper large-v3": "Multilingual, exact Hindi WER not isolated [source: https://huggingface.co/openai/whisper-large-v3]",
      "Google Speech-to-Text v2": "Hindi supported, exact public WER not captured [source: https://cloud.google.com/speech-to-text/v2/docs]",
      "Azure Speech": "Hindi supported, exact public WER not captured [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Deepgram Nova-2": "N/A (not publicly available) [source: https://developers.deepgram.com/docs/model]",
      "AssemblyAI Universal-2": "N/A (not publicly available) [source: https://www.assemblyai.com/docs]",
      "Amazon Transcribe": "N/A (not publicly available) [source: https://docs.aws.amazon.com/transcribe/]",
      "Sarvam AI Saarika": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "AI4Bharat IndicConformer": "N/A (not publicly available) [source: https://huggingface.co/ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large]",
      "Speechmatics Real-time": "N/A (not publicly available) [source: https://docs.speechmatics.com/]",
      "Kaldi / ESPnet": "N/A (not publicly available) [source: https://github.com/espnet/espnet]"
    },
    {
      "Metric": "Kannada support",
      "gummy-realtime-v1 / qwen3-asr-flash-realtime": "Project targets Kannada and keeps Kannada script [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "Whisper large-v3": "99 languages claimed; exact Kannada WER not isolated [source: https://huggingface.co/openai/whisper-large-v3]",
      "Google Speech-to-Text v2": "Yes (`kn-IN`) [source: https://cloud.google.com/speech-to-text/v2/docs]",
      "Azure Speech": "Yes (`kn-IN`) [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Deepgram Nova-2": "N/A (not publicly available) [source: https://developers.deepgram.com/docs/model]",
      "AssemblyAI Universal-2": "N/A (not publicly available) [source: https://www.assemblyai.com/docs]",
      "Amazon Transcribe": "N/A (not publicly available) [source: https://docs.aws.amazon.com/transcribe/]",
      "Sarvam AI Saarika": "Indic-market positioning [source: https://www.sarvam.ai/]",
      "AI4Bharat IndicConformer": "N/A (not publicly available) [source: https://huggingface.co/ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large]",
      "Speechmatics Real-time": "N/A (not publicly available) [source: https://docs.speechmatics.com/]",
      "Kaldi / ESPnet": "N/A (not publicly available) [source: https://github.com/espnet/espnet]"
    },
    {
      "Metric": "First partial transcript latency",
      "gummy-realtime-v1 / qwen3-asr-flash-realtime": "Same-session streaming path; exact public partial latency not disclosed [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "Whisper large-v3": "N/A (not publicly available) [source: https://huggingface.co/openai/whisper-large-v3]",
      "Google Speech-to-Text v2": "N/A (not publicly available) [source: https://cloud.google.com/speech-to-text/v2/docs]",
      "Azure Speech": "N/A (not publicly available) [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Deepgram Nova-2": "Streaming low-latency positioning in public docs [source: https://developers.deepgram.com/docs/model]",
      "AssemblyAI Universal-2": "N/A (not publicly available) [source: https://www.assemblyai.com/docs]",
      "Amazon Transcribe": "N/A (not publicly available) [source: https://docs.aws.amazon.com/transcribe/]",
      "Sarvam AI Saarika": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "AI4Bharat IndicConformer": "N/A (not publicly available) [source: https://huggingface.co/ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large]",
      "Speechmatics Real-time": "Realtime positioning [source: https://docs.speechmatics.com/]",
      "Kaldi / ESPnet": "N/A (not publicly available) [source: https://github.com/espnet/espnet]"
    },
    {
      "Metric": "Punctuation / capitalization",
      "gummy-realtime-v1 / qwen3-asr-flash-realtime": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "Whisper large-v3": "N/A (not publicly available) [source: https://huggingface.co/openai/whisper-large-v3]",
      "Google Speech-to-Text v2": "Documented punctuation features [source: https://cloud.google.com/speech-to-text/v2/docs]",
      "Azure Speech": "Supported in product family [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Deepgram Nova-2": "Entity, punctuation, capitalization improvements documented [source: https://developers.deepgram.com/docs/model]",
      "AssemblyAI Universal-2": "N/A (not publicly available) [source: https://www.assemblyai.com/docs]",
      "Amazon Transcribe": "N/A (not publicly available) [source: https://docs.aws.amazon.com/transcribe/]",
      "Sarvam AI Saarika": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "AI4Bharat IndicConformer": "N/A (not publicly available) [source: https://huggingface.co/ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large]",
      "Speechmatics Real-time": "N/A (not publicly available) [source: https://docs.speechmatics.com/]",
      "Kaldi / ESPnet": "N/A (not publicly available) [source: https://github.com/espnet/espnet]"
    },
    {
      "Metric": "Speaker diarization",
      "gummy-realtime-v1 / qwen3-asr-flash-realtime": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "Whisper large-v3": "N/A (not publicly available) [source: https://huggingface.co/openai/whisper-large-v3]",
      "Google Speech-to-Text v2": "Yes [source: https://cloud.google.com/speech-to-text/v2/docs]",
      "Azure Speech": "Yes [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Deepgram Nova-2": "Yes [source: https://developers.deepgram.com/docs/model]",
      "AssemblyAI Universal-2": "N/A (not publicly available) [source: https://www.assemblyai.com/docs]",
      "Amazon Transcribe": "Yes [source: https://docs.aws.amazon.com/transcribe/]",
      "Sarvam AI Saarika": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "AI4Bharat IndicConformer": "N/A (not publicly available) [source: https://huggingface.co/ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large]",
      "Speechmatics Real-time": "Yes [source: https://docs.speechmatics.com/]",
      "Kaldi / ESPnet": "N/A (not publicly available) [source: https://github.com/espnet/espnet]"
    },
    {
      "Metric": "Custom vocabulary / hotwords",
      "gummy-realtime-v1 / qwen3-asr-flash-realtime": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "Whisper large-v3": "N/A (not publicly available) [source: https://huggingface.co/openai/whisper-large-v3]",
      "Google Speech-to-Text v2": "Model adaptation / recognizer support [source: https://cloud.google.com/speech-to-text/v2/docs]",
      "Azure Speech": "Custom speech support [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Deepgram Nova-2": "Self-serve customization documented [source: https://developers.deepgram.com/docs/model]",
      "AssemblyAI Universal-2": "N/A (not publicly available) [source: https://www.assemblyai.com/docs]",
      "Amazon Transcribe": "Custom vocabulary support [source: https://docs.aws.amazon.com/transcribe/]",
      "Sarvam AI Saarika": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "AI4Bharat IndicConformer": "N/A (not publicly available) [source: https://huggingface.co/ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large]",
      "Speechmatics Real-time": "N/A (not publicly available) [source: https://docs.speechmatics.com/]",
      "Kaldi / ESPnet": "N/A (not publicly available) [source: https://github.com/espnet/espnet]"
    },
    {
      "Metric": "Noise robustness",
      "gummy-realtime-v1 / qwen3-asr-flash-realtime": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "Whisper large-v3": "HF card highlights robustness to accents/background noise [source: https://huggingface.co/openai/whisper-large-v3]",
      "Google Speech-to-Text v2": "N/A (not publicly available) [source: https://cloud.google.com/speech-to-text/v2/docs]",
      "Azure Speech": "N/A (not publicly available) [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Deepgram Nova-2": "Vendor claims strong real-world robustness [source: https://developers.deepgram.com/docs/model]",
      "AssemblyAI Universal-2": "N/A (not publicly available) [source: https://www.assemblyai.com/docs]",
      "Amazon Transcribe": "N/A (not publicly available) [source: https://docs.aws.amazon.com/transcribe/]",
      "Sarvam AI Saarika": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "AI4Bharat IndicConformer": "N/A (not publicly available) [source: https://huggingface.co/ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large]",
      "Speechmatics Real-time": "N/A (not publicly available) [source: https://docs.speechmatics.com/]",
      "Kaldi / ESPnet": "N/A (not publicly available) [source: https://github.com/espnet/espnet]"
    },
    {
      "Metric": "Price per hour of audio",
      "gummy-realtime-v1 / qwen3-asr-flash-realtime": "Project uses same-provider realtime stack; exact standalone ASR hourly conversion not computed [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "Whisper large-v3": "N/A (not publicly available) [source: https://huggingface.co/openai/whisper-large-v3]",
      "Google Speech-to-Text v2": "See pricing page from product docs [source: https://cloud.google.com/speech-to-text/v2/docs]",
      "Azure Speech": "See pricing page [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Deepgram Nova-2": "See pricing plans [source: https://developers.deepgram.com/docs/model]",
      "AssemblyAI Universal-2": "N/A (not publicly available) [source: https://www.assemblyai.com/docs]",
      "Amazon Transcribe": "N/A (not publicly available) [source: https://docs.aws.amazon.com/transcribe/]",
      "Sarvam AI Saarika": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "AI4Bharat IndicConformer": "N/A (not publicly available) [source: https://huggingface.co/ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large]",
      "Speechmatics Real-time": "N/A (not publicly available) [source: https://docs.speechmatics.com/]",
      "Kaldi / ESPnet": "N/A (not publicly available) [source: https://github.com/espnet/espnet]"
    },
    {
      "Metric": "On-device / edge deployment",
      "gummy-realtime-v1 / qwen3-asr-flash-realtime": "No, managed cloud path [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "Whisper large-v3": "Yes [source: https://huggingface.co/openai/whisper-large-v3]",
      "Google Speech-to-Text v2": "N/A (not publicly available) [source: https://cloud.google.com/speech-to-text/v2/docs]",
      "Azure Speech": "N/A (not publicly available) [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Deepgram Nova-2": "N/A (not publicly available) [source: https://developers.deepgram.com/docs/model]",
      "AssemblyAI Universal-2": "N/A (not publicly available) [source: https://www.assemblyai.com/docs]",
      "Amazon Transcribe": "N/A (not publicly available) [source: https://docs.aws.amazon.com/transcribe/]",
      "Sarvam AI Saarika": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "AI4Bharat IndicConformer": "Yes [source: https://huggingface.co/ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large]",
      "Speechmatics Real-time": "N/A (not publicly available) [source: https://docs.speechmatics.com/]",
      "Kaldi / ESPnet": "Yes [source: https://github.com/espnet/espnet]"
    }
  ],
  "criteria": [
    "streaming",
    "indic support",
    "same-platform",
    "operational simplicity"
  ],
  "fit_matrix": {
    "gummy-realtime-v1 / qwen3-asr-flash-realtime": [
      1,
      1,
      1,
      1
    ],
    "Whisper large-v3": [
      0,
      1,
      0,
      0.4
    ],
    "Google Speech-to-Text v2": [
      1,
      1,
      0,
      0.4
    ],
    "Azure Speech": [
      1,
      1,
      0,
      0.4
    ],
    "Deepgram Nova-2": [
      1,
      0.4,
      0,
      0.4
    ],
    "AssemblyAI Universal-2": [
      1,
      0,
      0,
      0.3
    ],
    "Amazon Transcribe": [
      1,
      0,
      0,
      0.3
    ],
    "Sarvam AI Saarika": [
      1,
      1,
      0,
      0.3
    ],
    "AI4Bharat IndicConformer": [
      0.5,
      1,
      0,
      0.3
    ],
    "Speechmatics Real-time": [
      1,
      0,
      0,
      0.3
    ],
    "Kaldi / ESPnet": [
      0.3,
      0,
      0,
      0.2
    ]
  }
}


In [ ]:
import pandas as pd
df = pd.DataFrame(data["comparison_rows"])
display(df)


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
    colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label == list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
    ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
labels=list(data['fit_matrix'].keys())
values=[sum(v) for v in data['fit_matrix'].values()]
fig,ax=plt.subplots(figsize=(12,5)); style(ax,"Ally Vision v2 — Category 6 - ASR Transcription Comparison Overall Fit Score")
ax.bar(labels,values,color=colors); ax.set_ylabel('Derived fit score',color='white'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.savefig(charts_dir / 'category6_asr_transcription_comparison_chart1.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
    colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label == list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
    ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
criteria=data['criteria']; selected=["gummy-realtime-v1 / qwen3-asr-flash-realtime", "Google Speech-to-Text v2", "Azure Speech", "Sarvam AI Saarika", "Whisper large-v3"]; x=np.arange(len(criteria)); width=0.8/len(selected)
fig,ax=plt.subplots(figsize=(12,5)); style(ax,"Ally Vision v2 — Category 6 - ASR Transcription Comparison Top-5 Criteria View")
for idx,label in enumerate(selected):
 vals=data['fit_matrix'][label]; color=ALLY if label==selected[0] else (DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else COMP); ax.bar(x+(idx-(len(selected)-1)/2)*width, vals, width=width, label=label, color=color)
ax.set_xticks(x); ax.set_xticklabels(criteria, rotation=20, ha='right', color='white'); ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white'); plt.tight_layout(); plt.savefig(charts_dir / 'category6_asr_transcription_comparison_chart2.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
    colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label == list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
    ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
criteria=data['criteria']; selected=["gummy-realtime-v1 / qwen3-asr-flash-realtime", "Google Speech-to-Text v2", "Azure Speech", "Sarvam AI Saarika", "Whisper large-v3"]; mat=np.array([data['fit_matrix'][k] for k in selected])
fig,ax=plt.subplots(figsize=(10,5)); ax.set_facecolor(BG); ax.figure.set_facecolor(BG); im=ax.imshow(mat,cmap='Blues',aspect='auto'); ax.set_title('Ally Vision v2 — Category 6 - ASR Transcription Comparison Trade-off Heatmap',color='white'); ax.set_xticks(np.arange(len(criteria))); ax.set_xticklabels(criteria, rotation=20, ha='right', color='white'); ax.set_yticks(np.arange(len(selected))); ax.set_yticklabels(selected,color='white')
for i in range(mat.shape[0]):
 for j in range(mat.shape[1]): ax.text(j,i,f"{mat[i,j]:.0f}",ha='center',va='center',color='white')
plt.colorbar(im); plt.tight_layout(); plt.savefig(charts_dir / 'category6_asr_transcription_comparison_chart3.png',dpi=150,bbox_inches='tight'); plt.show()


## Data Sources

| # | Source Name | URL | Value extracted |
|---|-------------|-----|-----------------|
| 1 | Whisper large-v3 HF card | https://huggingface.co/openai/whisper-large-v3 | multilingual card and open ASR leaderboard snippets |
| 2 | Google STT v2 docs | https://cloud.google.com/speech-to-text/v2/docs | streaming support and language coverage |
| 3 | Azure Speech language support | https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt | Hindi and Kannada support |
| 4 | Deepgram model docs | https://developers.deepgram.com/docs/model | streaming and voice-agent support |
| 5 | Project realtime client | https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py | current unified ASR integration path |


## CONCLUSION

The current ASR choice is strongest on integration shape, not just on raw benchmark claims. Ally Vision v2 uses one realtime session for speech input, speech output, interruption handling, and camera-grounded turns, so keeping ASR inside that same loop matters more than adopting a separate specialist provider by default.

→ Chosen for Ally Vision v2 ✅
